# Video decomposition test

In [1]:
from pathlib import Path
import numpy as np

from wavelet_utils import loadFilterParamDict, downscale_binary_video

In [2]:
temppath = r'D:\SynologyDriveSyncedDATA\PROCESSED\Waven'

libpath= Path(temppath) / 'gaborLibrary_5_3_8_5_4_5.npy'

paramspath = libpath.with_suffix('.json')   

videopath = Path(temppath) / 'zebra_s0_d420.0_fps59.94_RESAMPLED13fps.mp4'

In [3]:
library = np.load(libpath)
print(f"Loaded Gabor filter library from {libpath} with shape {library.shape}")

xs, ys, angles, sizes, freqs, phases, visual_coverage, full_screen_coverage, screen_x, screen_y = loadFilterParamDict(paramspath)

Loaded Gabor filter library from D:\SynologyDriveSyncedDATA\PROCESSED\Waven\gaborLibrary_5_3_8_5_4_5.npy with shape (5, 3, 8, 5, 4, 5, 100, 66)


In [4]:
print(f"full_screen_coverage: {full_screen_coverage}, visual_coverage: {visual_coverage}")

full_screen_coverage: [-90   0 -45  45], visual_coverage: [-90   0 -30  30]


## Downsample video

In [ ]:
downsampled_video_path=downscale_binary_video(videopath, full_screen_coverage, visual_coverage, screen_x, screen_y, force=True)

Input video: 5460 frames, 1280 x 1024
Crop pixels: x=0:1280, y=171:853
Output shape: (5460, 66, 100)


100%|██████████| 5460/5460 [00:12<00:00, 445.03it/s]

Saved: D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED13fps_downscaled.npy


## Display downsampled video result

In [6]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
from pathlib import Path

input_video_path = Path(videopath)
#output_npy_path 

out_video = np.load(downsampled_video_path, mmap_mode="r")

cap = cv2.VideoCapture(str(input_video_path))
n_frames = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), out_video.shape[0])

def show_frame(frame_idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret:
        print(f"Could not read frame {frame_idx}")
        return

    input_gray = frame[:, :, 0]
    output_frame = out_video[frame_idx]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(input_gray, cmap="gray")
    axes[0].set_title(f"Input frame {frame_idx}")
    axes[0].axis("off")

    axes[1].imshow(output_frame, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(f"Output frame {frame_idx}")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

interact(
    show_frame,
    frame_idx=IntSlider(
        value=0,
        min=0,
        max=n_frames - 1,
        step=1,
        description="Frame"
    )
)

interactive(children=(IntSlider(value=0, description='Frame', max=5459), Output()), _dom_classes=('widget-inte…

<function __main__.show_frame(frame_idx)>

## Decomposition